# 第 5 周练习 —— 进阶 RAG：LLM 分块预处理 + 重排 + 查询改写

## 练习目标

在「专业版」RAG 路线上练习四件事（摄取侧）：

1. **不用 LangChain**：原生 Python 最大灵活性
2. 用 **LLM** 按语义合理切块（chunking）
3. 沿用较好的块大小与 **embedding** 模型
4. 让 LLM 以更利于检索的方式重写块（文档预处理：headline + summary + original）

后面还会做：Chroma 入库、t-SNE 可视化、**rerank**、**query rewrite**、NER 关键词增强。

## 和本课第 5 周的关系

| 概念 | 本笔记本 |
|------|----------|
| 摄取 / 分块 | `fetch_documents` → `process_document` |
| 向量库 | Chroma `PersistentClient` + `text-embedding-3-large` |
| 高级检索 | `rerank` / `rewrite_query` / keyword 合并 |


In [ ]:
# ========== 导入与常量：模型 / Chroma 库名 / 知识库路径 / 块大小 ==========

# Path：知识库目录遍历
from pathlib import Path
# OpenAI：embeddings.create
from openai import OpenAI
# load_dotenv：读 .env 里的 API Key
from dotenv import load_dotenv
# Pydantic：结构化输出 Chunk / RankOrder
from pydantic import BaseModel, Field
# Chroma 持久化客户端
from chromadb import PersistentClient
# 进度条：批量 process_document 时显示
from tqdm import tqdm
# litellm.completion：统一调用聊天模型（含 response_format）
from litellm import completion
# 数值与可视化降维
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

# override=True：.env 覆盖已有环境变量
load_dotenv(override=True)

# 聊天 / 结构化输出用的模型 id（勿改）
MODEL = "gpt-4.1-nano"

# Chroma 持久化目录名与 collection 名
DB_NAME = "preprocessed_db"
collection_name = "docs"
# 嵌入模型 id
embedding_model = "text-embedding-3-large"
# 本地知识库根目录（按子文件夹区分 type）
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
# 估计切块数时用的平均字符规模
AVERAGE_CHUNK_SIZE = 500

# OpenAI 客户端实例（后面 embeddings 用）
openai = OpenAI()


In [ ]:
# ========== Result：模仿 LangChain Document 的轻量结构 ==========
# 受到 LangChain 文档的启发 - 让我们做一些类似的事情

# page_content：检索/展示用正文；metadata：来源与类型等
class Result(BaseModel):
    page_content: str
    metadata: dict


In [ ]:
# ========== Chunk / Chunks：LLM 结构化分块的 schema ==========
# 完美代表块的类：标题 + 摘要 + 原文，便于检索与回答

class Chunk(BaseModel):
    # Field description 会进 JSON schema，供模型理解字段含义（英文保留）
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    # 拼成 Result：headline/summary/原文 用空行连接；metadata 来自原文档
    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


# 外层包装：一次响应返回多个 Chunk
class Chunks(BaseModel):
    chunks: list[Chunk]


## 三步摄取流水线

1. 从知识库读取文档（自制版 DirectoryLoader）
2. 调用 LLM 把文档转成结构化块（Chunks）
3. 把块的向量存进 Chroma

就是这样！

### 先从第 1 步开始


In [ ]:
# ========== 第 1 步：fetch_documents —— 遍历 knowledge-base/**/*.md ==========

def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    # 一级子目录名当作文档 type（products / employees / ...）
    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        # 递归找所有 Markdown
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents


In [ ]:
# 执行第 1 步：加载全部知识库文档到内存列表
documents = fetch_documents()


### 很好！进入第 2 步 —— 用 LLM 制作块（chunking + 预处理）


In [ ]:
# ========== make_prompt：为单篇文档生成「请切成重叠块」的用户提示 ==========

def make_prompt(document):
    # 按平均块大小粗估应切成几块，写进提示引导模型
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    # 下面整段 f-string 是发给模型的英文指令（可运行 prompt，勿翻译）
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""


In [ ]:
# 预览：打印第一篇文档生成的切块提示（检查 how_many 与正文是否进 prompt）
print(make_prompt(documents[0]))


In [ ]:
# ========== make_messages：把 prompt 包成 litellm 需要的 messages 列表 ==========

def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]


In [ ]:
# 预览：第一篇文档的 messages 结构（尚未真正调用 API）
make_messages(documents[0])


In [ ]:
# ========== process_document：LLM 结构化输出 → Chunks → List[Result] ==========

def process_document(document):
    # 组装 user messages
    messages = make_messages(document)
    # response_format=Chunks：要求模型按 Pydantic schema 返回 JSON
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    # 取出助手回复字符串
    reply = response.choices[0].message.content
    # 校验并解析为 Chunks，再取出 chunks 列表
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    # 每个 Chunk 转成带 metadata 的 Result
    return [chunk.as_result(document) for chunk in doc_as_chunks]


In [ ]:
# 试跑：只处理第一篇文档，观察返回的 Result 列表
process_document(documents[0])


In [ ]:
# ========== create_chunks：对全部文档串行 process，tqdm 显示进度 ==========

def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks


In [ ]:
# 执行第 2 步：批量 LLM 分块（可能较慢 / 消耗额度）
chunks = create_chunks(documents)


In [ ]:
# 看一共生成了多少个预处理块
print(len(chunks))


### 嗯，这很简单！如果慢一点也正常。

在 Python 模块版本里，作者用多进程池并行加速；
若遇到 **rate limit**，可以关掉并行。

### 最后，第 3 步 —— 保存嵌入到 Chroma


In [ ]:
# ========== 第 3 步：create_embeddings —— 批量向量化并写入 Chroma ==========

def create_embeddings(chunks):
    # 打开（或创建）持久化 Chroma 目录
    chroma = PersistentClient(path=DB_NAME)
    # 若同名 collection 已存在则先删，保证本次重建干净
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    # 取出要嵌入的文本（已含 headline/summary/原文）
    texts = [chunk.page_content for chunk in chunks]
    # OpenAI embeddings 批量接口
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    # 新建/取得 collection
    collection = chroma.get_or_create_collection(collection_name)

    # id 用序号字符串；metadata 原样写入
    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")


In [ ]:
# 执行第 3 步：把全部 chunks 向量化入库
create_embeddings(chunks)


# 摄取做完了……对吧？

等等！可视化还没上——用 t-SNE 看看向量空间是否按文档类型聚簇。


In [ ]:
# ========== 从 Chroma 读回 embeddings / documents / metadatas，准备可视化 ==========

# 重新打开库并取得 collection
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
# include 指定要取回的字段（含向量）
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
# 每条的 type 字段，用于上色
doc_types = [metadata['type'] for metadata in metadatas]
# 四种 type 映射到四种颜色（顺序与列表 index 对应）
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]


In [ ]:
# ========== 2D t-SNE + Plotly 散点：观察类型聚类 ==========

# 降到 2 维；random_state 固定可复现
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# 创建二维散点图：颜色=文档类型，hover 显示类型与文本前 100 字
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()


In [ ]:
# ========== 3D t-SNE + Plotly Scatter3d：再看一版空间结构 ==========

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# 创建 3D 散点图
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()


## 现在 —— 构建高级 RAG

将使用这些技术：

1. **重新排名（rerank）**——对检索结果再排序
2. **查询重写（query rewrite）**——把用户问题改写成更易命中知识库的短问


In [ ]:
# ========== RankOrder：让 LLM 返回「按相关度排列的 chunk id 列表」 ==========

class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )


In [ ]:
# ========== rerank：用 LLM 按问题相关度重排检索到的 chunks ==========

def rerank(question, chunks):
    # system：规定「只返回重排后的 chunk id 列表」（英文 prompt 勿改）
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    # user：带上问题 + 编号后的各 chunk 正文
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    # CHUNK ID 从 1 起编号，与后面 order 里的整数对应
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    # 结构化输出 RankOrder
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    # order 里是 1-based id → 转回 chunks 列表下标
    return [chunks[i - 1] for i in order]


In [ ]:
# ========== fetch_context_unranked：向量检索 Top-K，不做重排 ==========

# 默认取 10 条候选，供后续 rerank
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    # 问题 → embedding
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    # Chroma 近邻查询
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    # 把 documents + metadatas 打成 Result 列表
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks


In [ ]:
# ========== NER 实验：用 transformers pipeline 从问题抽关键词 ==========

# 两道示例问题（英文问句保持原样，会进检索/模型）
question = "Who won the IIOTY award?"
question2 = "Who went to Manchester University?"

from transformers import pipeline

# 初始化 NER pipeline；模型 id 勿改
# 'dslim/bert-base-NER' 是一般英语 NER 的常用选择
nlp = pipeline("ner", model="dslim/bert-base-NER", tokenizer="dslim/bert-base-NER")
# 对 question2 做命名实体识别
ner_results = nlp(question2)
# 取出识别到的 word 片段作为 keywords
keywords = [result['word'] for result in ner_results]
# 打印结果，观察能否抽出 Manchester / University 等
print(keywords)


In [ ]:
# 笔记本调试残留：查看 Chunks 类型对象本身（可忽略或当检查导入）
Chunks


In [ ]:
# 再取一遍 NER 的 word 列表（与上一格 keywords 同类）
key_words = [result['word'] for result in ner_results]


In [ ]:
# 用例：IIOTY 奖项问题 → 未重排的检索结果
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)


In [ ]:
# 浏览未重排结果：每条只打印正文前 15 字
for chunk in chunks:
    print(chunk.page_content[:15]+"...")


In [ ]:
# 对同一批 chunks 做 LLM rerank
reranked = rerank(question, chunks)


In [ ]:
# 对比：重排后的前 15 字预览（顺序应更贴问题）
for chunk in reranked:
    print(chunk.page_content[:15]+"...")


In [ ]:
# 更难的用例：Manchester University；加大 K=20，看原文命中落在第几位
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)


In [ ]:
# 对 Manchester 问题做 rerank
reranked = rerank(question, chunks)


In [ ]:
# 重排后：含 manchester 的块排到了第几（理想情况应靠前）
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)


In [ ]:
# 查看重排后最相关那一条的完整 page_content
reranked[0].page_content


In [ ]:
# ========== fetch_context：先向量检索，再 LLM rerank（组合拳） ==========

def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)


In [ ]:
# ========== SYSTEM_PROMPT：最终回答阶段的系统提示（含 {context} 占位） ==========
# 英文模板保持原样；后面 .format(context=...) 注入检索片段

SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""


In [ ]:
# ========== make_rag_messages：拼 system(含来源) + history + 当前 user ==========
# 在上下文中，包括块的来源

def make_rag_messages(question, history, chunks):
    # 每段前面标注 Extract from <source>
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]


In [ ]:
# ========== rewrite_query：把用户问题改写成更短、更易检索的查询 ==========

def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    # 整段英文指令：要求只输出一条 refined question（勿翻译）
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    # 注意：这里把整段指令放在 system role
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content


In [ ]:
# 试跑查询改写：IIOTY 问题、空历史
rewrite_query("Who won the IIOTY award?", [])


In [ ]:
# ========== answer_question：改写 → 检索+重排 → 组装 messages → 生成答案 ==========

def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    # 先改写成检索用短问
    query = rewrite_query(question, history)
    print(query)
    # fetch_context = 向量 Top-K + rerank
    chunks = fetch_context(query)
    # 用「原始用户问题」进最终对话，不用改写后的 query
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks


In [ ]:
# 端到端试跑：IIOTY 奖项
answer_question("Who won the IIOTY award?", [])


In [ ]:
# 端到端试跑：Manchester University（观察改写+重排是否帮上忙）
answer_question("Who went to Manchester University?", [])


In [ ]:
# ========== fetch_keyword_chunks：在全文里做关键词子串匹配，补向量检索漏网 ==========

def fetch_keyword_chunks(keywords, max_results=5):
    """Search vector store for chunks containing any of the keywords."""
    matched_chunks = []
    # 同一 source 只收一次，避免重复刷屏
    seen_sources = set()
    for keyword in keywords:
        # 取出库内全部 documents（简单实现；数据量大时需改）
        results = collection.get(include=["documents", "metadatas"])
        for doc, meta in zip(results["documents"], results["metadatas"]):
            meta_dict = dict(meta)  # Convert Mapping to dict
            # 大小写不敏感子串匹配 + source 去重
            if keyword.lower() in doc.lower() and meta_dict["source"] not in seen_sources:
                matched_chunks.append(Result(page_content=doc, metadata=meta_dict))
                seen_sources.add(meta_dict["source"])
                if len(matched_chunks) >= max_results:
                    break
        if len(matched_chunks) >= max_results:
            break
    return matched_chunks

# 通过关键字上下文增强答案管道（见下一格 answer_question_with_keywords）


In [ ]:
# 试跑关键词检索：Manchester + University
fetch_keyword_chunks(["Manchester", "University"],)


In [ ]:
# ========== answer_question_with_keywords：向量 RAG + NER 关键词块合并后回答 ==========

def answer_question_with_keywords(question: str, history=None) -> tuple[str, list]:
    """
    Answer a question using RAG, adding keyword-matched chunks as extra context.
    """
    # 默认空历史，避免可变默认参数陷阱
    if history is None:
        history = []
    # 提取关键词（复用前面的 NER pipeline）
    ner_results = nlp(question)
    keywords = [result['word'] for result in ner_results]
    # 获取正常上下文：改写查询 + fetch_context（含 rerank）
    query = rewrite_query(question, history)
    chunks = fetch_context(query)
    # 获取关键词块
    keyword_chunks = fetch_keyword_chunks(keywords)
    # 合并并删除重复（以 page_content 为键）
    all_chunks = {c.page_content: c for c in chunks}
    for kc in keyword_chunks:
        all_chunks[kc.page_content] = kc
    combined_chunks = list(all_chunks.values())
    messages = make_rag_messages(question, history, combined_chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, combined_chunks
